In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)
n = 5000

txns = pd.DataFrame({
    "txn_id": range(1, n + 1),
    "txn_date": pd.to_datetime("2026-01-01") + pd.to_timedelta(np.random.randint(0, 300, n), unit="D"),
    "station": np.random.choice(["Johor Bahru", "Kuala Lumpur", "Penang", "Ipoh"], n),
    "fuel_type": np.random.choice(["RON95", "RON97", "DIESEL"], n, p=[0.6, 0.15, 0.25]),
    "channel": np.random.choice(["Card", "App", "Cash"], n, p=[0.5, 0.3, 0.2]),
    "litres": np.random.uniform(10, 60, n).round(2)
})

print(len(txns))
print(txns.head())

5000
   txn_id   txn_date       station fuel_type channel  litres
0       1 2026-04-13   Johor Bahru     RON95    Card   54.94
1       2 2026-09-28        Penang     RON95    Cash   48.68
2       3 2026-04-17  Kuala Lumpur    DIESEL    Cash   53.56
3       4 2026-03-13          Ipoh     RON95     App   20.70
4       5 2026-07-08        Penang     RON95    Cash   39.78


In [2]:
prices = pd.read_csv("../data/conform/fuel_prices_long.csv", parse_dates=["date"])

txns = txns.sort_values("txn_date")
prices = prices.sort_values("date")

enriched = pd.merge_asof(
    txns, prices,
    left_on="txn_date", right_on="date",
    by="fuel_type", direction="backward"
)

enriched["amount_rm"] = (enriched["litres"] * enriched["price_rm"]).round(2)

print(len(enriched))
print(enriched["price_rm"].isna().sum(), "missing prices")
print(enriched.head())

5000
0 missing prices
   txn_id   txn_date       station fuel_type channel  litres       date  \
0     537 2026-01-01  Kuala Lumpur     RON95    Card   11.51 2026-01-01   
1    4929 2026-01-01          Ipoh     RON95    Card   37.19 2026-01-01   
2     700 2026-01-01        Penang     RON95    Cash   45.62 2026-01-01   
3    1621 2026-01-01        Penang     RON97    Card   39.99 2026-01-01   
4     848 2026-01-01        Penang     RON95    Card   34.29 2026-01-01   

   price_rm  amount_rm  
0      2.56      29.47  
1      2.56      95.21  
2      2.56     116.79  
3      3.16     126.37  
4      2.56      87.78  


In [ ]:
enrich_dir = Path("../data/enrich")
enrich_dir.mkdir(parents=True, exist_ok=True)

daily = (enriched
    .groupby([enriched["txn_date"].dt.date, "station", "channel"])
    .agg(num_txns=("txn_id", "count"),
         total_litres=("litres", "sum"),
         total_rm=("amount_rm", "sum"))
    .reset_index()
    .rename(columns={"txn_date": "business_date"}))

enriched.to_csv(enrich_dir / "fact_fuel_transactions.csv", index=False)
daily.to_csv(enrich_dir / "agg_daily_station_channel.csv", index=False)

print(len(daily), "daily rows")
print(daily["num_txns"].sum(), "transactions in summary")